In [ ]:
# =============================================================================
# Cell Annotation (QC, doublet removal, integration, clustering)
# =============================================================================

library(Seurat)
library(DoubletFinder)
library(celda)
library(harmony)
library(tidyr)
library(dplyr)

# -----------------------------------------------------------------------------
# Helper function to detect doublets using DoubletFinder
# -----------------------------------------------------------------------------
Find_doublet <- function(data) {
  # Parameter sweep to find optimal pK
  sweep.res.list <- paramSweep_v3(data, PCs = 1:30, sct = FALSE)
  sweep.stats <- summarizeSweep(sweep.res.list, GT = FALSE)
  bcmvn <- find.pK(sweep.stats)
  
  # Expected number of doublets (5% of cells)
  nExp_poi <- round(0.05 * ncol(data))
  
  # Select pK with maximum MeanBC
  p <- as.numeric(as.vector(bcmvn[bcmvn$MeanBC == max(bcmvn$MeanBC), "pK"]))
  
  # Run DoubletFinder
  data <- doubletFinder_v3(data, PCs = 1:30, pN = 0.25, pK = p,
                           nExp = nExp_poi, reuse.pANN = FALSE, sct = FALSE)
  
  # Rename the doublet classification column
  colnames(data@meta.data)[ncol(data@meta.data)] <- "doublet_info"
  return(data)
}

# -----------------------------------------------------------------------------
# Load individual studies and perform QC, doublet removal, and decontamination
# -----------------------------------------------------------------------------
study <- list('Leng', 'Gerrits', 'OteroGarcia', 'Bakken', 'Hodge')
sc.list <- lapply(study, function(x) {
  # Load the RData file (assumes file named e.g. "Leng.RData" contains a Seurat object 'sc')
  load(paste0(x, '.RData'))
  sc$'orig.ident' <- x
  
  # Calculate mitochondrial percentage
  sc$percent.mt <- PercentageFeatureSet(sc, pattern = "^MT-")
  
  # Basic QC filtering
  sc <- subset(sc,
               nFeature_RNA > 200 &
               nCount_RNA > 500 &
               nCount_RNA < quantile(sc$nCount_RNA, 0.95) &
               percent.mt < 10)
  
  # Normalize and find variable features
  sc <- NormalizeData(sc)
  sc <- FindVariableFeatures(sc, selection.method = "vst", nfeatures = 3000)
  sc <- ScaleData(sc)
  sc <- RunPCA(sc)
  sc <- RunUMAP(sc, dims = 1:30)
  
  # Doublet detection
  sc <- Find_doublet(sc)
  sc <- subset(sc, subset = doublet_info == "Singlet")
  
  # Remove temporary pANN columns
  c <- grep("pANN_", colnames(sc@meta.data))
  if (length(c) > 0) sc@meta.data <- sc@meta.data[, -c]
  
  # DecontX to estimate RNA contamination
  decontx_result <- decontX(sc@assays$RNA@counts)
  sc$Contamination <- decontx_result$contamination
  sc <- subset(sc, Contamination < 0.2)
  
  return(sc)
})
names(sc.list) <- study

# -----------------------------------------------------------------------------
# Merge all datasets and integrate with Harmony
# -----------------------------------------------------------------------------
combined <- merge(sc.list[[1]], sc.list[-1])

# SCTransform normalization, regressing out nCount_RNA
combined <- SCTransform(combined,
                        vars.to.regress = c("nCount_RNA"),
                        variable.features.n = 3000,
                        verbose = FALSE)

# PCA
combined <- RunPCA(combined, npcs = 50)

# Harmony integration to remove batch effects by 'orig.ident'
combined <- RunHarmony(combined, group.by.vars = "orig.ident")

# UMAP on Harmony dimensions
combined <- RunUMAP(combined, reduction = "harmony", dims = 1:30, reduction.name = 'harmony.umap')

# Clustering
combined <- FindNeighbors(combined, reduction = "harmony", dims = 1:30)
combined <- FindClusters(combined, resolution = 0.5)

# -----------------------------------------------------------------------------
# Manual annotation of clusters based on marker genes
# -----------------------------------------------------------------------------
# Find markers for each cluster (optional, can be used to guide annotation)
markers <- FindAllMarkers(combined, only.pos = TRUE)

# Manual assignment of cell types to clusters (using a recode-like approach)
# Note: 'rscode' is assumed to be a custom recoding function; here we use dplyr::recode.
# The mapping below is based on prior knowledge or marker inspection.
combined$celltype <- dplyr::recode(combined$seurat_clusters,
    '0'  = 'MIC',
    '1'  = 'OLG',
    '2'  = 'MIC',
    '3'  = 'AST',
    '4'  = 'AST',
    '5'  = 'MIC',
    '6'  = 'EXC',
    '7'  = 'sc',          # likely a typo, maybe "SC" (smooth muscle?) – keep as in original
    '8'  = 'Fibroblast',
    '9'  = 'EXC',
    '10' = 'AST',
    '11' = 'EXC',
    '12' = 'EXC',
    '13' = 'EXC',
    '14' = 'T',
    '15' = 'INC',
    '16' = 'AST',
    '17' = 'EXC',
    '18' = 'Unknown',
    '19' = 'Pericyte',
    '20' = 'INC',
    '21' = 'OLG',
    '22' = 'INC',
    '23' = 'MIC',
    '24' = 'EXC',
    '25' = 'INC',
    '26' = 'MIC',
    '27' = 'AST',
    '28' = 'Unknown',
    '29' = 'OLG',
    '30' = 'Macrophage',
    '31' = 'EXC',
    '32' = 'OPC',
    '33' = 'MIC')

# Save the integrated and annotated object
save(combined, file = 'integrated_sc.RData')

library(Seurat)
library(ggplot2)

# Define output directory (hidden)
output_dir <- "path/to/output/folder"
dir.create(output_dir, recursive = TRUE, showWarnings = FALSE)

# Define color palettes for each metadata category
color_lists <- list(
  "orig.ident" = c("Bakken" = "#E31A1C", "Gerrits" = "#7D8000", "Hodge" = "#008B45", 
                    "Leng" = "#0072B2", "OteroGarcia" = "#911EB4"),
  "region"     = c("M1" = "#D62728", "MTG" = "#8C564B", "OC" = "#2CA02C", 
                    "OTC" = "#1F77B4", "PFC" = "#9467BD", "SFG" = "#E377C2"),
  "sex"        = c("F" = "#E41A1C", "M" = "#377EB8"),
  "pheno"      = c("AD" = "#D62728", "non-AD" = "#1F77B4"),
  "new_celltype" = c("AST" = "#800000", "ENDO" = "#006400", "EXC" = "#FF0000", "FIB" = "#00008B", 
                      "INH" = "#4B0082", "MAC" = "#008B8B", "MIC" = "#9400D3", "OLG" = "#C71585", 
                      "OPC" = "#8B4513", "PER" = "#5F9EA0", "TC" = "#DAA520")
)

# Loop over categories and save plots
for (category in names(color_lists)) {
  p <- DimPlot(seurat_obj, reduction = "harmony.umap", group.by = category, 
               cols = color_lists[[category]], pt.size = 0.8,
               label = (category == "new_celltype"), label.size = 5, repel = TRUE) +
    labs(title = category, x = "UMAP1", y = "UMAP2") +
    theme_classic() + theme(aspect.ratio = 1)
  
  ggsave(file.path(output_dir, paste0(category, "_plot.pdf")), plot = p, width = 7, height = 6)
}